# Day 4 — Joining Two Datasets

**Theme:** A single dataset answers one kind of question. Two datasets joined together answer the interesting ones.

**By the end of this notebook you will have:**
- Loaded a PLUTO property sample alongside your clean 311 data
- Merged them on a shared key (zip code)
- Answered: which Brooklyn zip codes have the most complaints *and* the highest assessed property values?

**The math teacher analogy for merge:**  
Imagine you have two rosters — one with student names and test scores, one with student names and attendance records. A merge matches them on the shared column (student name) and produces one combined table. Same idea here: zip code is the shared column.

**No hints unless you ask.**

In [8]:
import pandas as pd

df_311 = '../data/311_cleaned.csv'
df_pluto = '../data/pluto.csv'
# Store them in df_311 and df_pluto.


## Step 1 — Inspect both DataFrames

Before merging, you need to know:
1. What are the column names in each?
2. What does the zip code column look like in each — same format, same name?

In [17]:


df_311 = pd.read_csv("../data/311_cleaned.csv")
#print(df_311.columns.tolist())
print(df_311.head(3))


   Unique Key            Created Date             Closed Date Agency  \
0    69189924  05/31/2026 11:44:40 PM  06/01/2026 02:08:25 AM   NYPD   
1    69195629  05/31/2026 11:44:28 PM  06/01/2026 02:08:47 AM   NYPD   
2    69188434  05/31/2026 11:43:59 PM  06/01/2026 01:29:05 AM   NYPD   

                       Agency Name Problem (formerly Complaint Type)  \
0  New York City Police Department                   Illegal Parking   
1  New York City Police Department       Non-Emergency Police Matter   
2  New York City Police Department                 Abandoned Vehicle   

  Problem Detail (formerly Descriptor) Additional Details  \
0         Commercial Overnight Parking                NaN   
1                          Trespassing                NaN   
2                   With License Plate                NaN   

                Location Type  Incident Zip  ... Bridge Highway Name  \
0             Street/Sidewalk       11214.0  ...                 NaN   
1  Residential Building/House    

In [24]:
df_pluto = pd.read_csv('../data/pluto.csv')
print(df_pluto.head(3))

/var/folders/01/7mmm8p_55mz_znpc8yd21gjc0000gn/T/ipykernel_38767/1340933164.py:1: DtypeWarning: Columns (0: zonedist3, 1: zonedist4, 2: overlay2, 3: spdist1, 4: spdist2, 5: ltdheight, 6: splitzone, 7: factryarea, 8: numbldgs, 9: unitsres, 10: unitstotal, 11: lotfront, 12: lotdepth, 13: bldgfront, 14: bldgdepth, 15: irrlotcode, 16: histdist, 17: landmark, 18: condono, 19: edesignum, 20: dcpedited, 21: mihopt1, 22: mihopt2, 23: mihopt3, 24: mihopt4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_pluto = pd.read_csv('../data/pluto.csv')


  borough  Tax block  Tax lot  community board  census tract 2010  cb2010  \
0      BX       5480      111            210.0              160.0  1009.0   
1      BX       5480      121            210.0              160.0  1009.0   
2      BX       5472      109            210.0              160.0  1006.0   

   schooldist  council district  postcode firecomp  ...  \
0         8.0              13.0   10465.0     E072  ...   
1         8.0              13.0   10465.0     E072  ...   
2         8.0              13.0   10465.0     E072  ...   

                       transitzone  geom  basempdate dcasdate edesigdate  \
0  Beyond the Greater Transit Zone   NaN         NaN      NaN        NaN   
1  Beyond the Greater Transit Zone   NaN         NaN      NaN        NaN   
2  Beyond the Greater Transit Zone   NaN         NaN      NaN        NaN   

  landmkdate masdate polidate rpaddate zoningdate  
0        NaN     NaN      NaN      NaN        NaN  
1        NaN     NaN      NaN      NaN       

## Step 2 — Check the key column

A merge fails silently if the key columns don't match exactly — same values, same data type.
Check the dtype of the zip code column in each DataFrame before you merge.

In the 311 data the column is called `Incident Zip`. In PLUTO it's `ZipCode`.

If one is an integer and the other is a string, they won't match and your merged table will be empty.

In [43]:
import pandas as pd

df = pd.read_csv('../data/pluto.csv')
df2 = pd.read_csv('../data/311_cleaned.csv')


print(df['postcode'].dtype)
print(df2['Incident Zip'].dtype)




/var/folders/01/7mmm8p_55mz_znpc8yd21gjc0000gn/T/ipykernel_38767/3646442995.py:3: DtypeWarning: Columns (0: zonedist3, 1: zonedist4, 2: overlay2, 3: spdist1, 4: spdist2, 5: ltdheight, 6: splitzone, 7: factryarea, 8: numbldgs, 9: unitsres, 10: unitstotal, 11: lotfront, 12: lotdepth, 13: bldgfront, 14: bldgdepth, 15: irrlotcode, 16: histdist, 17: landmark, 18: condono, 19: edesignum, 20: dcpedited, 21: mihopt1, 22: mihopt2, 23: mihopt3, 24: mihopt4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/pluto.csv')


float64
float64


## Step 3 — Aggregate 311 by zip code

PLUTO has one row per *property*. Your 311 data has one row per *complaint*.
You can't merge them directly — you first need to summarize 311 down to one row per zip code (a complaint count).

Use `groupby` on `Incident Zip` and count the rows. Store the result as a DataFrame called `zip_counts`.

Gotcha: after a `groupby().size()`, the result is a Series with the zip as the index. Call `.reset_index()` on it to turn it back into a regular DataFrame with columns you can merge on. Name the count column `complaint_count`.

In [99]:
import pandas as pd

df2 = pd.read_csv('../data/311_cleaned.csv')

zip_counts = df2.groupby('Incident Zip').size()
#print(zip_counts)

complaint_count = df2.groupby('Incident Zip').size().reset_index(name='complaint_count')
print(complaint_count)


    Incident Zip  complaint_count
0        11201.0             4395
1        11203.0             2494
2        11204.0             3054
3        11205.0             2230
4        11206.0             3210
5        11207.0             3888
6        11208.0             3888
7        11209.0             2905
8        11210.0             1954
9        11211.0             3173
10       11212.0             2971
11       11213.0             2628
12       11214.0             3675
13       11215.0             3156
14       11216.0             2924
15       11217.0             2682
16       11218.0             3098
17       11219.0             1921
18       11220.0             2845
19       11221.0             3687
20       11222.0             2796
21       11223.0             2985
22       11224.0             1285
23       11225.0             2866
24       11226.0             4876
25       11228.0             1245
26       11229.0             2605
27       11230.0             3148
28       11231

## Step 4 — Aggregate PLUTO by zip code

Similarly, PLUTO has one row per property. Summarize it to one row per zip code.
Calculate the **median assessed value** per zip — median is more robust than mean when a few luxury buildings could skew the average.

Store the result as `zip_pluto` with columns: `ZipCode`, `median_assessed`.

In [102]:
import pandas as pd
df1 = pd.read_csv('../data/pluto.csv')

zip_counts2 = df1.groupby('postcode').size().reset_index(name='lot_count')

print(zip_counts2)







/var/folders/01/7mmm8p_55mz_znpc8yd21gjc0000gn/T/ipykernel_38767/2909588090.py:2: DtypeWarning: Columns (0: zonedist3, 1: zonedist4, 2: overlay2, 3: spdist1, 4: spdist2, 5: ltdheight, 6: splitzone, 7: unitsres, 8: unitstotal, 9: lotfront, 10: lotdepth, 11: bldgfront, 12: bldgdepth, 13: irrlotcode, 14: histdist, 15: landmark, 16: condono, 17: edesignum, 18: dcpedited, 19: mihopt1, 20: mihopt2, 21: mihopt3, 22: mihopt4) have mixed types. Specify dtype option on import or set low_memory=False.
  df1 = pd.read_csv('../data/pluto.csv')


    postcode  lot_count
0    11201.0       3322
1    11203.0      11054
2    11204.0       9955
3    11205.0       3144
4    11206.0       4579
5    11207.0      11400
6    11208.0      11269
7    11209.0       8470
8    11210.0       8573
9    11211.0       5418
10   11212.0       6499
11   11213.0       5062
12   11214.0       9499
13   11215.0       9001
14   11216.0       6070
15   11217.0       3987
16   11218.0       6867
17   11219.0       8693
18   11220.0       9011
19   11221.0       9143
20   11222.0       5272
21   11223.0      10307
22   11224.0       2819
23   11225.0       3939
24   11226.0       5313
25   11228.0       7867
26   11229.0      12050
27   11230.0       7954
28   11231.0       4831
29   11232.0       2960
30   11233.0       7986
31   11234.0      19382
32   11235.0       7923
33   11236.0      14680
34   11237.0       4359
35   11238.0       4846
36   11239.0        657
37   11241.0          1
38   11249.0       1537
39   11251.0          2
40   11414.0    

## Step 5 — Merge

Now both tables have one row per zip code. Merge them.

The syntax is:
```python
pd.merge(left_df, right_df, left_on='left_key_column', right_on='right_key_column')
```

Your key columns have different names (`Incident Zip` vs `ZipCode`) so you need both `left_on` and `right_on`.

In [111]:
import pandas as pd
df2 = pd.read_csv('../data/311_cleaned.csv')
df1 = pd.read_csv('../data/pluto.csv')

## Step 5 — Merge

merged = pd.merge(complaint_count, zip_counts2, left_on='Incident Zip', right_on='postcode')

print(merged)
# Store as merged.
# Print the result.


/var/folders/01/7mmm8p_55mz_znpc8yd21gjc0000gn/T/ipykernel_38767/3677472984.py:3: DtypeWarning: Columns (0: zonedist3, 1: zonedist4, 2: overlay2, 3: spdist1, 4: spdist2, 5: ltdheight, 6: splitzone, 7: unitsres, 8: unitstotal, 9: lotfront, 10: lotdepth, 11: bldgfront, 12: bldgdepth, 13: irrlotcode, 14: histdist, 15: landmark, 16: condono, 17: edesignum, 18: dcpedited, 19: mihopt1, 20: mihopt2, 21: mihopt3, 22: mihopt4) have mixed types. Specify dtype option on import or set low_memory=False.
  df1 = pd.read_csv('../data/pluto.csv')


    Incident Zip  complaint_count  postcode  lot_count
0        11201.0             4395   11201.0       3322
1        11203.0             2494   11203.0      11054
2        11204.0             3054   11204.0       9955
3        11205.0             2230   11205.0       3144
4        11206.0             3210   11206.0       4579
5        11207.0             3888   11207.0      11400
6        11208.0             3888   11208.0      11269
7        11209.0             2905   11209.0       8470
8        11210.0             1954   11210.0       8573
9        11211.0             3173   11211.0       5418
10       11212.0             2971   11212.0       6499
11       11213.0             2628   11213.0       5062
12       11214.0             3675   11214.0       9499
13       11215.0             3156   11215.0       9001
14       11216.0             2924   11216.0       6070
15       11217.0             2682   11217.0       3987
16       11218.0             3098   11218.0       6867
17       1

## Step 6 — Answer the question

Which zip codes have the most complaints **and** the highest assessed values?

Sort `merged` by `complaint_count` descending and look at what `median_assessed` is doing alongside it.
Is there a pattern?

In [112]:
print(merged.sort_values('complaint_count', ascending=False))



    Incident Zip  complaint_count  postcode  lot_count
24       11226.0             4876   11226.0       5313
0        11201.0             4395   11201.0       3322
5        11207.0             3888   11207.0      11400
6        11208.0             3888   11208.0      11269
19       11221.0             3687   11221.0       9143
12       11214.0             3675   11214.0       9499
35       11238.0             3212   11238.0       4846
4        11206.0             3210   11206.0       4579
9        11211.0             3173   11211.0       5418
13       11215.0             3156   11215.0       9001
27       11230.0             3148   11230.0       7954
32       11235.0             3124   11235.0       7923
16       11218.0             3098   11218.0       6867
2        11204.0             3054   11204.0       9955
21       11223.0             2985   11223.0      10307
10       11212.0             2971   11212.0       6499
14       11216.0             2924   11216.0       6070
7        1

## Day 4 Recap

| What you did | Why it matters |
|---|---|
| Aggregated before merging | You can only join tables that are at the same grain (one row per zip) |
| Checked key dtypes | A type mismatch is the #1 silent merge failure |
| `pd.merge()` | The core tool for combining any two datasets that share a column |
| Median vs mean | Robust summary when outliers (luxury buildings) exist |

### What's coming on Day 5

We'll put the merged data on a **map** using geopandas. You'll see your zip code complaint counts as a choropleth — the kind of visualization that shows up in every urban planning report.

## Step 5 — Merge

Now both tables have one row per zip code. Merge them.

The syntax is:
```python
pd.merge(left_df, right_df, left_on='left_key_column', right_on='right_key_column')
```

Your key columns have different names (`Incident Zip` vs `ZipCode`) so you need both `left_on` and `right_on`.

## Step 5 — Merge

Now both tables have one row per zip code. Merge them.

The syntax is:
```python
pd.merge(left_df, right_df, left_on='left_key_column', right_on='right_key_column')
```

Your key columns have different names (`Incident Zip` vs `ZipCode`) so you need both `left_on` and `right_on`.